# LAB 10 — Tokenizer Alignment & Offset Mapping

Notebook này thực hiện:
1. Tokenize (question, context) pair
2. Kiểm tra input_ids, attention_mask, offset_mapping
3. Character span → Token span alignment
4. Xử lý truncation và sliding window

In [1]:
import torch
from transformers import AutoTokenizer

MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

C:\Users\thanh\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Tokenization của QA

Input format: `[CLS] question tokens [SEP] context tokens [SEP]`

In [2]:
question = "What is the capital of France?"
context = "The capital of France is Paris."

# Tokenize với return_offsets_mapping
encoding = tokenizer(
    question,
    context,
    return_offsets_mapping=True,
    return_token_type_ids=True,
    padding=False,
    truncation=False
)

tokens = tokenizer.convert_ids_to_tokens(encoding["input_ids"])

print("=== Tokenization Result ===")
print(f"Tokens: {tokens}")
print(f"\ninput_ids: {encoding['input_ids']}")
print(f"\nattention_mask: {encoding['attention_mask']}")
print(f"\ntoken_type_ids: {encoding['token_type_ids']}")
print(f"\noffset_mapping: {encoding['offset_mapping']}")

# Hiển thị bảng token
print("\n=== Token Details ===")
print(f"{'Idx':<5} {'Token':<15} {'Type':<6} {'Offset'}")
print("-" * 45)
for i, (tok, tid, offset) in enumerate(zip(tokens, encoding['token_type_ids'], encoding['offset_mapping'])):
    segment = "Q" if tid == 0 else "C"
    print(f"{i:<5} {tok:<15} {segment:<6} {offset}")

=== Tokenization Result ===
Tokens: ['[CLS]', 'what', 'is', 'the', 'capital', 'of', 'france', '?', '[SEP]', 'the', 'capital', 'of', 'france', 'is', 'paris', '.', '[SEP]']

input_ids: [101, 2054, 2003, 1996, 3007, 1997, 2605, 1029, 102, 1996, 3007, 1997, 2605, 2003, 3000, 1012, 102]

attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

token_type_ids: [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1]

offset_mapping: [(0, 0), (0, 4), (5, 7), (8, 11), (12, 19), (20, 22), (23, 29), (29, 30), (0, 0), (0, 3), (4, 11), (12, 14), (15, 21), (22, 24), (25, 30), (30, 31), (0, 0)]

=== Token Details ===
Idx   Token           Type   Offset
---------------------------------------------
0     [CLS]           Q      (0, 0)
1     what            Q      (0, 4)
2     is              Q      (5, 7)
3     the             Q      (8, 11)
4     capital         Q      (12, 19)
5     of              Q      (20, 22)
6     france          Q      (23, 29)
7     ?               Q      (29, 30)
8  

## 2. Character Span → Token Span Alignment

Dataset cung cấp `answer_start` (character position) và `answer_text`.  
Cần chuyển từ character span sang token span bằng offset_mapping.

In [3]:
def find_token_span(encoding, answer_start_char, answer_text, tokenizer):
    """
    Chuyển character span sang token span sử dụng offset_mapping.
    
    Args:
        encoding: output từ tokenizer
        answer_start_char: vị trí character bắt đầu của answer trong context
        answer_text: text của answer
        tokenizer: tokenizer instance
    
    Returns:
        (start_token, end_token) hoặc (None, None) nếu answer bị truncated
    """
    answer_end_char = answer_start_char + len(answer_text)
    offsets = encoding['offset_mapping']
    token_type_ids = encoding.get('token_type_ids', [0] * len(offsets))
    
    start_token = None
    end_token = None
    
    for token_idx, (start, end) in enumerate(offsets):
        # Chỉ xét context tokens (token_type_id == 1 cho BERT)
        if token_type_ids[token_idx] == 0:
            continue
        # Bỏ qua special tokens (offset = (0,0))
        if start == 0 and end == 0:
            continue
            
        if start <= answer_start_char < end:
            start_token = token_idx
        if start < answer_end_char <= end:
            end_token = token_idx
    
    return start_token, end_token


# Test: Answer = "Paris"
answer_text = "Paris"
answer_start_char = context.index(answer_text)
print(f"Context: '{context}'")
print(f"Answer: '{answer_text}'")
print(f"Character span: ({answer_start_char}, {answer_start_char + len(answer_text)})")

start_tok, end_tok = find_token_span(encoding, answer_start_char, answer_text, tokenizer)
print(f"Token span: start={start_tok}, end={end_tok}")
if start_tok is not None and end_tok is not None:
    answer_tokens = tokens[start_tok:end_tok+1]
    print(f"Answer tokens: {answer_tokens}")
    print(f"Decoded: {tokenizer.decode(encoding['input_ids'][start_tok:end_tok+1])}")

Context: 'The capital of France is Paris.'
Answer: 'Paris'
Character span: (25, 30)
Token span: start=14, end=14
Answer tokens: ['paris']
Decoded: paris


In [4]:
# Test: Answer = "capital of France"
answer_text2 = "capital of France"
answer_start_char2 = context.index(answer_text2)
print(f"Context: '{context}'")
print(f"Answer: '{answer_text2}'")
print(f"Character span: ({answer_start_char2}, {answer_start_char2 + len(answer_text2)})")

start_tok2, end_tok2 = find_token_span(encoding, answer_start_char2, answer_text2, tokenizer)
print(f"Token span: start={start_tok2}, end={end_tok2}")
if start_tok2 is not None and end_tok2 is not None:
    answer_tokens2 = tokens[start_tok2:end_tok2+1]
    print(f"Answer tokens: {answer_tokens2}")
    print(f"Decoded: {tokenizer.decode(encoding['input_ids'][start_tok2:end_tok2+1])}")

Context: 'The capital of France is Paris.'
Answer: 'capital of France'
Character span: (4, 21)
Token span: start=10, end=12
Answer tokens: ['capital', 'of', 'france']
Decoded: capital of france


## 3. Truncation & Sliding Window

Khi context dài hơn max_length, cần sử dụng sliding window với stride.

In [5]:
# Ví dụ context dài
long_context = (
    "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. "
    "It is named after the engineer Gustave Eiffel, whose company designed and built the tower. "
    "Locally nicknamed La dame de fer, it was constructed from 1887 to 1889 as the centerpiece "
    "of the 1889 World's Fair. Although initially criticized by some of France's leading artists "
    "and intellectuals for its design, it has since become a global cultural icon of France and "
    "one of the most recognizable structures in the world. The Eiffel Tower is the most visited "
    "paid monument in the world with 6.91 million visitors in 2015. The tower is 330 metres tall, "
    "about the same height as an 81-storey building, and the tallest structure in Paris."
)

long_question = "How tall is the Eiffel Tower?"

# Tokenize với sliding window
inputs = tokenizer(
    long_question,
    long_context,
    truncation="only_second",
    max_length=128,          # nhỏ để demo sliding window
    stride=64,
    return_overflowing_tokens=True,
    return_offsets_mapping=True
)

print(f"Số windows: {len(inputs['input_ids'])}")
for i, (ids, offsets) in enumerate(zip(inputs['input_ids'], inputs['offset_mapping'])):
    toks = tokenizer.convert_ids_to_tokens(ids)
    print(f"\n=== Window {i+1} ===")
    print(f"Tokens ({len(toks)}): {toks[:10]} ... {toks[-5:]}")
    print(f"Decoded context: {tokenizer.decode(ids)[:200]}...")

Số windows: 2

=== Window 1 ===
Tokens (128): ['[CLS]', 'how', 'tall', 'is', 'the', 'e', '##iff', '##el', 'tower', '?'] ... ['the', 'most', 'visited', 'paid', '[SEP]']
Decoded context: [CLS] how tall is the eiffel tower? [SEP] the eiffel tower is a wrought - iron lattice tower on the champ de mars in paris, france. it is named after the engineer gustave eiffel, whose company designe...

=== Window 2 ===
Tokens (88): ['[CLS]', 'how', 'tall', 'is', 'the', 'e', '##iff', '##el', 'tower', '?'] ... ['million', 'visitors', 'in', '2015', '[SEP]']
Decoded context: [CLS] how tall is the eiffel tower? [SEP] from 1887 to 1889 as the centerpiece of the 1889 world ' s fair. although initially criticized by some of france ' s leading artists and intellectuals for its...


In [6]:
# Kiểm tra overflow_to_sample_mapping
print("=== overflow_to_sample_mapping ===")
print(f"Mapping: {inputs.get('overflow_to_sample_mapping', 'N/A')}")
print("\nGiải thích: overflow_to_sample_mapping cho biết mỗi window thuộc sample nào")
print("trong batch. Khi xử lý nhiều samples cùng lúc, mapping này giúp")
print("truy ngược từ window về sample gốc.")

=== overflow_to_sample_mapping ===
Mapping: [0, 0]

Giải thích: overflow_to_sample_mapping cho biết mỗi window thuộc sample nào
trong batch. Khi xử lý nhiều samples cùng lúc, mapping này giúp
truy ngược từ window về sample gốc.


## 4. Xử lý các trường hợp đặc biệt

In [7]:
# Trường hợp 1: Answer bắt đầu giữa token (subword)
context_subword = "The unhappiness of the people was evident."
question_subword = "What was evident?"
answer_subword = "unhappiness"

enc_sub = tokenizer(question_subword, context_subword, return_offsets_mapping=True, return_token_type_ids=True)
toks_sub = tokenizer.convert_ids_to_tokens(enc_sub['input_ids'])
print("=== Subword case ===")
print(f"Tokens: {toks_sub}")
print(f"Offsets: {enc_sub['offset_mapping']}")

start_char_sub = context_subword.index(answer_subword)
s, e = find_token_span(enc_sub, start_char_sub, answer_subword, tokenizer)
print(f"Answer '{answer_subword}' → token span: ({s}, {e})")
if s and e:
    print(f"Answer tokens: {toks_sub[s:e+1]}")

=== Subword case ===
Tokens: ['[CLS]', 'what', 'was', 'evident', '?', '[SEP]', 'the', 'un', '##ha', '##pp', '##iness', 'of', 'the', 'people', 'was', 'evident', '.', '[SEP]']
Offsets: [(0, 0), (0, 4), (5, 8), (9, 16), (16, 17), (0, 0), (0, 3), (4, 6), (6, 8), (8, 10), (10, 15), (16, 18), (19, 22), (23, 29), (30, 33), (34, 41), (41, 42), (0, 0)]
Answer 'unhappiness' → token span: (7, 10)
Answer tokens: ['un', '##ha', '##pp', '##iness']


In [8]:
# Trường hợp 2: Answer bị truncation
print("\n=== Truncation case ===")
# Dùng context dài, answer ở cuối, max_length nhỏ
answer_long = "330 metres"
answer_start_long = long_context.index(answer_long)

enc_trunc = tokenizer(
    long_question, long_context,
    truncation="only_second",
    max_length=64,  # rất ngắn → answer bị cắt
    return_offsets_mapping=True,
    return_token_type_ids=True
)

s_tr, e_tr = find_token_span(enc_trunc, answer_start_long, answer_long, tokenizer)
print(f"Answer '{answer_long}' character start: {answer_start_long}")
print(f"Token span: ({s_tr}, {e_tr})")
if s_tr is None or e_tr is None:
    print("→ Answer bị truncated! Không tìm thấy token span.")
    print("Giải pháp: Sử dụng sliding window (return_overflowing_tokens=True) với stride.")


=== Truncation case ===
Answer '330 metres' character start: 619
Token span: (None, None)
→ Answer bị truncated! Không tìm thấy token span.
Giải pháp: Sử dụng sliding window (return_overflowing_tokens=True) với stride.


In [9]:
# Trường hợp 3: Answer nằm trong window nào khi dùng sliding window?
print("\n=== Sliding window — tìm answer trong đúng window ===")
inputs_sw = tokenizer(
    long_question, long_context,
    truncation="only_second",
    max_length=128,
    stride=64,
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
    return_token_type_ids=True
)

for win_idx in range(len(inputs_sw['input_ids'])):
    enc_win = {
        'offset_mapping': inputs_sw['offset_mapping'][win_idx],
        'token_type_ids': inputs_sw['token_type_ids'][win_idx],
        'input_ids': inputs_sw['input_ids'][win_idx]
    }
    s_w, e_w = find_token_span(enc_win, answer_start_long, answer_long, tokenizer)
    if s_w is not None and e_w is not None:
        toks_w = tokenizer.convert_ids_to_tokens(enc_win['input_ids'])
        print(f"Window {win_idx+1}: answer found at token span ({s_w}, {e_w})")
        print(f"Answer tokens: {toks_w[s_w:e_w+1]}")
    else:
        print(f"Window {win_idx+1}: answer NOT in this window")


=== Sliding window — tìm answer trong đúng window ===
Window 1: answer NOT in this window
Window 2: answer NOT in this window


## Tóm tắt

| Concept | Giải thích |
|---------|------------|
| Character index | Vị trí ký tự trong chuỗi gốc (0-based) |
| Word index | Vị trí từ trong câu (tách bằng space) |
| Token index | Vị trí token sau khi tokenize (bao gồm subword, special tokens) |
| Offset mapping | Ánh xạ từ token index → (char_start, char_end) trong chuỗi gốc |
| Stride | Độ overlap giữa các window khi context bị chia nhỏ |